# Chapter 3 · Deutsch-Jozsa Algorithm

## Objectives

1. Understand the oracle-based computation model and the problem solved by Deutsch-Jozsa.
2. Build constant and balanced oracles for $n$ bits.
3. Implement the complete algorithm in Qiskit and verify the quantum-classical separation.

---

## 3.1 The Deutsch-Jozsa Problem

Given an oracle that computes a function $f: \{0,1\}^n \to \{0,1\}$, it is guaranteed that $f$ is:

- **Constant**: $f(x) = 0$ or $f(x) = 1$ for all $x$, or
- **Balanced**: $f(x) = 0$ for exactly half of the inputs and $f(x) = 1$ for the other half.

A classical algorithm needs $2^{n-1} + 1$ evaluations in the worst case. The Deutsch-Jozsa quantum algorithm determines the answer with **a single oracle call**.

The algorithm applies the sequence:

$$|0\rangle^{\otimes n}|1\rangle \xrightarrow{H^{\otimes n+1}} |+\rangle^{\otimes n}|-\rangle \xrightarrow{U_f} \cdots \xrightarrow{H^{\otimes n}} \text{measure}$$

If the result is $|00\ldots 0\rangle$, $f$ is constant; otherwise, it is balanced.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Qiskit and local modules loaded.')

## 3.2 Oracles for Deutsch-Jozsa

In [ ]:
def oracle_constant_0(n: int) -> QuantumCircuit:
    """Oracle for f(x) = 0 (constant). Does nothing."""
    return QuantumCircuit(n + 1, name='Uf(const-0)')


def oracle_constant_1(n: int) -> QuantumCircuit:
    """Oracle for f(x) = 1 (constant). Applies X to ancilla qubit."""
    qc = QuantumCircuit(n + 1, name='Uf(const-1)')
    qc.x(n)
    return qc


def oracle_balanced(n: int, balanced_string: str | None = None) -> QuantumCircuit:
    """Oracle for a balanced function.

    The function f(x) = CNOT based on a bitstring: applies CNOTs from
    each input qubit whose bit is '1' in balanced_string to the ancilla.
    If balanced_string is None, '1'*n is used (all active).

    Parameters
    ----------
    n : int
        Number of input qubits.
    balanced_string : str, optional
        Binary string of length n. '1' at position i activates the CNOT.
    """
    if balanced_string is None:
        balanced_string = '1' * n
    assert len(balanced_string) == n, 'balanced_string must have length n'

    qc = QuantumCircuit(n + 1, name='Uf(balanced)')
    for i, bit in enumerate(balanced_string):
        if bit == '1':
            qc.cx(i, n)
    return qc


# Visualize the oracles for n=3
n = 3
print('Constant oracle 0:')
print(oracle_constant_0(n).draw('text'))
print('\nConstant oracle 1:')
print(oracle_constant_1(n).draw('text'))
print('\nBalanced oracle (string=101):')
print(oracle_balanced(n, '101').draw('text'))

## 3.3 Complete Deutsch-Jozsa Algorithm

In [ ]:
def deutsch_jozsa(oracle: QuantumCircuit, n: int) -> QuantumCircuit:
    """Builds the complete Deutsch-Jozsa circuit.

    Parameters
    ----------
    oracle : QuantumCircuit
        Oracle circuit U_f (n+1 qubits).
    n : int
        Number of input qubits.

    Returns
    -------
    QuantumCircuit
        Complete circuit ready to execute.
    """
    qc = QuantumCircuit(n + 1, n)

    # Initialization: ancilla in |1〉
    qc.x(n)
    qc.barrier(label='Init')

    # Hadamard over all qubits
    qc.h(range(n + 1))
    qc.barrier(label='H⊗(n+1)')

    # Oracle
    qc.compose(oracle, inplace=True)
    qc.barrier(label='Oracle')

    # Second Hadamard layer over input qubits
    qc.h(range(n))
    qc.barrier(label='H⊗n')

    # Measurement of input qubits
    qc.measure(range(n), range(n))

    return qc


# Instantiate with the oracles
n = 4
backend = AerSimulator()

for oracle_fn, oracle_name in [
    (oracle_constant_0(n), 'Constant 0'),
    (oracle_constant_1(n), 'Constant 1'),
    (oracle_balanced(n, '1011'), 'Balanced (1011)'),
    (oracle_balanced(n, '0110'), 'Balanced (0110)'),
]:
    qc = deutsch_jozsa(oracle_fn, n)
    job = backend.run(qc, shots=1024)
    counts = job.result().get_counts()
    all_zeros = '0' * n
    decision = 'CONSTANT' if all_zeros in counts and counts[all_zeros] == 1024 else 'BALANCED'
    print(f'Oracle: {oracle_name:25s} → Decision: {decision}')
    print(f'  Counts: {counts}')

In [ ]:
# Circuit visualization for n=3
n = 3
qc_example = deutsch_jozsa(oracle_balanced(n, '110'), n)
print(f'Deutsch-Jozsa circuit, n={n}, balanced oracle (110):')
print(qc_example.draw('text'))

## 3.4 Scale and Quantum Advantage

The following empirical analysis shows that the quantum algorithm always needs exactly **one** oracle evaluation, regardless of $n$.

In [ ]:
import random

classical_queries = []
quantum_queries   = []
n_values = range(1, 12)

for n in n_values:
    # Classical (worst case)
    classical_queries.append(2 ** (n - 1) + 1)
    # Quantum: always 1
    quantum_queries.append(1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(n_values), classical_queries, 'o-', color='#f78166',
        label='Classical (worst case)', linewidth=2)
ax.plot(list(n_values), quantum_queries, 's-', color='#58a6ff',
        label='Quantum (Deutsch-Jozsa)', linewidth=2)
ax.set_xlabel('Number of input bits (n)')
ax.set_ylabel('Oracle evaluations')
ax.set_title('Quantum advantage: Deutsch-Jozsa vs. classical algorithm')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 3.5 Proposed Exercises

1. Implement the $n=1$ case (original Deutsch) manually with NumPy matrices, without Qiskit. Verify the result.

2. What happens if the oracle $U_f$ introduces errors (bit-flip with probability $p = 0.01$)? Modify the circuit to include a noise channel and analyze the error rate of the algorithm.

3. Build a balanced oracle for $n=5$ in which exactly half of the input strings map to 1. Verify with the algorithm that it is correctly detected.

4. Formally prove (with state calculations) that measuring $|00\ldots 0\rangle$ with probability 1 implies that $f$ is constant.